# Hate Speech Analysis

This notebook combines EDA and TF-IDF + LinearSVC training for the final submission.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import LinearSVC

from src.preprocess import LABEL_MAP, build_clean_dataframe, load_raw_dataset

sns.set_theme(style="whitegrid")

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
VECTORIZER_PATH = MODELS_DIR / "tfidf_vectorizer.joblib"
MODEL_PATH = MODELS_DIR / "hate_speech_svc.joblib"

df_raw = load_raw_dataset()
df = build_clean_dataframe(df_raw)

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

## Exploratory Data Analysis

We inspect class balance and tweet length patterns before training.

In [ ]:
class_counts = df["class"].value_counts().sort_index()
class_df = pd.DataFrame({
    "class": class_counts.index,
    "label": [LABEL_MAP.get(i, str(i)) for i in class_counts.index],
    "count": class_counts.values,
    "percent": (class_counts.values / len(df) * 100).round(2),
})
class_df

In [ ]:
plt.figure(figsize=(8, 4))
ax = sns.barplot(data=class_df, x="label", y="count", hue="label", dodge=False, legend=False)
ax.set_title("Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
df["char_len"] = df["tweet"].astype(str).str.len()
df["word_len"] = df["tweet"].astype(str).str.split().str.len()
df[["char_len", "word_len"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["char_len"], bins=50, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Character Length Distribution")
sns.histplot(df["word_len"], bins=40, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Word Length Distribution")
plt.tight_layout()
plt.show()

## TF-IDF + LinearSVC Training

We train a tuned LinearSVC model because it performed best among the classical approaches for this dataset.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_tweet"],
    df["class"],
    test_size=0.2,
    random_state=42,
    stratify=df["class"],
)

vectorizer = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Train matrix:", X_train.shape)
print("Test matrix:", X_test.shape)

In [ ]:
param_grid = {
    "C": [0.1, 0.5, 1.0, 2.0],
    "class_weight": ["balanced", {0: 3, 1: 1, 2: 1.5}],
    "loss": ["hinge", "squared_hinge"],
}

grid = GridSearchCV(
    estimator=LinearSVC(random_state=42, max_iter=15000),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=1,
    verbose=1,
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print("Best params:", grid.best_params_)
print(f"Best CV f1_macro: {grid.best_score_:.4f}")

In [ ]:
y_pred = best_model.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
}

pd.DataFrame([metrics]).round(4)

In [ ]:
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

In [ ]:
labels = sorted(df["class"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{i}" for i in labels], columns=[f"pred_{i}" for i in labels])
cm_df

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - LinearSVC")
plt.tight_layout()
plt.show()

In [ ]:
joblib.dump(vectorizer, VECTORIZER_PATH)
joblib.dump(best_model, MODEL_PATH)
print(f"Saved vectorizer to {VECTORIZER_PATH}")
print(f"Saved model to {MODEL_PATH}")